In [ ]:
# GPT-2-Large: finetune_vq ONLY. Skips the ~12h train_baseline+run_experiments
# stages by reusing the already-trained baseline.pt from a Kaggle Dataset
# (that earlier run completed Baseline/PTQ/QAT-INT8 fully on matched 20M-char
# data, but got session-limit-cancelled mid-way through this exact step).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone -q https://github.com/abdurrahmanrussel/QAT-VQ-Compression.git repo
%cd repo
!git checkout -q gpt2medium-wikitext103-qatvq
!git log --oneline -3
!echo "--- /kaggle/input contents ---"
!find /kaggle/input -maxdepth 3 2>&1
!cat gpt2_scaling/artifacts/gpt2-large/results.json

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.18" scikit-learn matplotlib bitsandbytes

In [ ]:
%cd /kaggle/working/repo/gpt2_scaling
import os, shutil, glob
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
try:
    import bitsandbytes as bnb
    print("bitsandbytes OK, version", bnb.__version__)
except Exception as e:
    print("bitsandbytes NOT available:", e)

# find baseline.pt anywhere under /kaggle/input (don't assume the exact
# mount path -- the assumed path failed twice, this is more robust)
candidates = glob.glob("/kaggle/input/**/baseline.pt", recursive=True)
print("found candidates:", candidates)
assert candidates, "baseline.pt not found anywhere under /kaggle/input -- dataset did not mount"
src = candidates[0]

os.makedirs("artifacts/gpt2-large", exist_ok=True)
shutil.copy(src, "artifacts/gpt2-large/baseline.pt")
print("staged baseline.pt:", os.path.getsize("artifacts/gpt2-large/baseline.pt") / 1e9, "GB")

## Codebook fine-tune only
Seed **2** (not the notebook default of 1 used previously) -- confirmed via
the original run's seed-search log as the actual best-of-3 seed (16.65 ppl
vs 16.79/16.85 for seeds 0/1). Same settings as the interrupted run
otherwise: 20M-char matched subset, 2 epochs, bs=1 (8-bit AdamW needed).

In [ ]:
!python finetune_vq.py --model gpt2-large --seed 2 --epochs 2 --lr 5e-6 --bs_train 1 --bs_eval 1 --train_subset_chars 20000000

In [ ]:
!python make_figures.py
!cat artifacts/scaling_table.md

In [ ]:
# Push results back to GitHub. Requires a Kaggle Secret named GITHUB_TOKEN
# (fine-grained PAT, repo=QAT-VQ-Compression, Contents: Read and write) --
# add it via this kernel's editor -> Add-ons -> Secrets before running.
# If this fails (Kaggle's internal secrets service has been flaky on every
# run so far), results are still recoverable via `kaggle kernels output`.
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

import subprocess
def sh(cmd):
    print('$', cmd.replace(token, '***') if token in cmd else cmd)
    subprocess.run(cmd, shell=True, check=True)

sh('git config user.email "abdurrahmanrussel77@gmail.com"')
sh('git config user.name "Md Abdur Rahman"')
sh('git add artifacts/gpt2-large/results.json artifacts/gpt2-large/results_table.md '
   'artifacts/gpt2-large/figures/ artifacts/scaling_table.md artifacts/figures/scaling_comparison.png')
sh('git commit -m "Complete GPT-2-Large QAT+VQ codebook finetune (seed 2, matched data)" || echo "nothing to commit"')
sh(f'git push https://{token}@github.com/abdurrahmanrussel/QAT-VQ-Compression.git gpt2medium-wikitext103-qatvq')